In [4]:
import os
import joblib 
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [5]:
MODEL_FILE = "full_model.pkl"
PIPELINE_FILE = 'pipeline.pkl'

def build_pipeline(num_attribs, cat_attribs):
    # For numerical columns
    num_pipline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # For categorical columns
    cat_pipline = Pipeline([ 
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # full pipeline
    full_pipeline = ColumnTransformer([
        ("num", num_pipline, num_attribs), 
        ('cat', cat_pipline, cat_attribs)
    ])

    return full_pipeline

In [9]:
if not os.path.exists(MODEL_FILE):

    housing = pd.read_csv("housing.csv")

    # stratified test set
    housing['income_cat'] = pd.cut(housing["median_income"], 
                                bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf], 
                                labels=[1, 2, 3, 4, 5])

    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

    for train_index, test_index in split.split(housing, housing['income_cat']):
        housing.loc[test_index].drop("income_cat", axis=1).to_csv("input.csv", index=False) 
        housing = housing.loc[train_index].drop("income_cat", axis=1)  

    housing = housing.copy()

    # Feature Engineering
    housing["total_bedrooms"].fillna(housing["total_bedrooms"].median(), inplace=True)

    housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
    housing["population_per_household"] = housing["population"] / housing["households"]
    housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]

    housing_labels = housing["median_house_value"].copy()
    housing_features = housing.drop("median_house_value", axis=1)
    
    num_attribs = housing_features.drop("ocean_proximity", axis=1).columns.tolist()
    cat_attribs = ["ocean_proximity"]

    pipeline = build_pipeline(num_attribs, cat_attribs) 

    full_model = Pipeline([
        ("preprocessing", pipeline),
        ("model", RandomForestRegressor(random_state=42))
    ])


    param_grid = {
        "model__n_estimators": [100, 200], # number of trees
        "model__max_depth": [10, 20, None], 
        "model__min_samples_split": [2, 5]  #min samples needed
    }

    grid_search = GridSearchCV(
        full_model,
        param_grid,
        cv=3,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    grid_search.fit(housing_features, housing_labels)

    full_model = grid_search.best_estimator_
    
    full_model.fit(housing_features, housing_labels)

    joblib.dump(full_model, "full_model.pkl")
    print("Model is trained.")

    scores = cross_val_score(
    full_model,
    housing_features,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=5
    )
    print("CV RMSE:", -scores.mean())

else:
    # inference
    model = joblib.load("full_model.pkl")

    input_data = pd.read_csv('input.csv')

    # Apply SAME feature engineering
    input_data["total_bedrooms"].fillna(
    input_data["total_bedrooms"].median(), inplace=True
    )
    input_data["rooms_per_household"] = input_data["total_rooms"] / input_data["households"]
    input_data["bedrooms_per_room"] = input_data["total_bedrooms"] / input_data["total_rooms"]
    input_data["population_per_household"] = input_data["population"] / input_data["households"]

    predictions = model.predict(input_data)
    input_data['median_house_value'] = predictions

    input_data.to_csv("output.csv", index=False)
    print("Inference is complete, results saved to output.csv ")

/var/folders/s8/yx760cxn7dd7tm40jbtj81800000gn/T/ipykernel_82152/1594505595.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  housing["total_bedrooms"].fillna(housing["total_bedrooms"].median(), inplace=True)


Model is trained.
CV RMSE: 50502.69596011462
